# w9_flash.ipynb — @512 STRUCTURE BLITZ (user top priority)

The full 3×4-ish CE/I-attachment matrix (10 cells) at anchor 512, drained across **every GPU
on this pod** in parallel: {no-I, I×2} × {per-view CE, expander CE, pooled
CE, pool→expander CE}. Results decide which structure the scaling theory is
built on. Cells whose result json already exists skip (ce@512 is done);
claims are heartbeat-compatible with the campaign pods, so l40/a100 pods
running the shared table will not double-run anything. AUTO-STOPS when the
matrix is complete.

User rule encoded in the worker: I always attaches after the multi-views,
and AFTER the expander whenever one exists (i2expce/i2poolexpce pay I in
E-space; deployed space of expander arms receives no direct loss).


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

# the eight matrix cells (arm, cap) -- done cells skip automatically
FLASH = [
    # I in DEPLOYED space (i2*; i2exp* = ORIGINAL n4expce design)
    ("wcle_i2ce_icetf", 512),            # I2@dep + per-view CE@dep
    ("wcle_i2expce_icetf", 512),         # I2@dep + per-view CE@E (original)
    ("wcle_i2poolce_icetf", 512),        # I2@dep + pooled CE@dep
    # I in E-space (expi2* = NEW design)
    ("wcle_ceexpi2_icetf", 512),         # per-view CE@dep + I2@E
    ("wcle_poolceexpi2_icetf", 512),     # pooled CE@dep + I2@E
    ("wcle_expi2expce_icetf", 512),      # DUAL E: I@E_I + CE@E_CE
    ("wcle_expi2poolexpce_icetf", 512),  # DUAL E: I@E_I + pool->E_CE->CE
    ("wcle_shexpi2ce_icetf", 512),       # SHARED E: I@E + CE@E
    ("wcle_shexpi2poolce_icetf", 512),   # SHARED E: I@E + pool->E->CE
    # no I
    ("wcle_ce_cetf", 512),               # per-view CE@dep (DONE, skips)
    ("wcle_expce_cetf", 512),            # per-view CE@E
    # direction-closure wave (pool position + exp/cmp combos)
    ("wcle_i2poolexpce_icetf", 512),     # I2@dep + pool->exp->CE
    ("wcle_i2poolcmpce_icetf", 512),     # I2@dep + pool->cmp->CE
    ("wcle_expi2cmpce_icetf", 512),      # DUAL: I2@exp + CE@cmp
    ("wcle_cmpi2expce_icetf", 512),      # DUAL: I2@cmp + CE@exp
    ("wcle_cmpi2cmpce_icetf", 512),      # DUAL: I2@cmp + CE@cmp
    ("wcle_expi2poolcmpce_icetf", 512),  # DUAL: I2@exp + pool->cmp->CE
    ("wcle_shexpi2poolexpce_icetf", 512),  # SHARED exp: pool-BEFORE-E CE
    ("wcle_shcmpi2poolcmpce_icetf", 512),  # SHARED cmp: pool-BEFORE-E CE
    # symmetry completion (final 30)
    ("wcle_poolcmpce_cetf", 512),        # no-I pool->cmp->CE
    ("wcle_cecmpi2_icetf", 512),         # CE@dep per-view + I2@cmp
    ("wcle_poolcecmpi2_icetf", 512),     # CE@dep pooled + I2@cmp
    ("wcle_shcmpi2poolce_icetf", 512),   # SHARED cmp, pool-AFTER-E
    ("wcle_cmpi2poolexpce_icetf", 512),  # DUAL: I2@cmp + pool->exp->CE
    ("wcle_cmpi2poolcmpce_icetf", 512),  # DUAL: I2@cmp + pool->cmp->CE
    # DOWN-projector (cmp, 128->128->64, SimCLR direction)
    ("wcle_cmpce_cetf", 512),            # no-I CE@cmp
    ("wcle_i2cmpce_icetf", 512),         # I2@dep + CE@cmp
    ("wcle_shcmpi2ce_icetf", 512),       # SHARED cmp: I2@cmp + CE@cmp
    ("wcle_poolce_cetf", 512),           # pooled CE@dep
    ("wcle_poolexpce_cetf", 512),        # pool->E->CE
]
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(FLASH)} matrix cells")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Drain the matrix across ALL GPUs (heartbeat claims; done cells skip).
import os, queue, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
jobs = queue.Queue()
for arm, cap in FLASH:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / J.result_name(nm)).exists():
        print(f"[skip] {nm} done"); continue
    jobs.put((arm, cap, nm))
fails = []

def worker(gpu):
    while True:
        try:
            arm, cap, nm = jobs.get_nowait()
        except queue.Empty:
            return
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True)
            continue
        log = logd / f"{arm}_g{cap}.log"
        cmd = ["python", "-u", J.FS_WORKER,
               "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
               "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(J.FS_EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        print(f"[gpu{gpu}] start {nm}", flush=True)
        t0 = time.time()
        with open(log, "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))
        if p.returncode != 0:
            fails.append((nm, str(log)))
        print(f"[gpu{gpu}] " + ("ok" if p.returncode == 0 else "FAIL")
              + f" {nm} [{(time.time()-t0)/60:.1f} min]", flush=True)

stop_evt = threading.Event()
mon = threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True)
mon.start()
gpus = J.detect_gpus()
threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
stop_evt.set()
print(f"FLASH drained in {(time.time()-t0)/3600:.1f} h; {len(fails)} failed")
for nm, lg in fails:
    print("  FAILED:", nm, "->", lg)


In [ ]:
# Readout: the 2x4 matrix.
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
GRID = [("no-I", [("ce_cetf", "per-view"), ("expce_cetf", "expander"),
                  ("poolce_cetf", "pooled"), ("poolexpce_cetf", "pool->E")]),
        ("I2  ", [("i2ce_icetf", "per-view"), ("i2expce_icetf", "expander"),
                  ("i2poolce_icetf", "pooled"), ("i2poolexpce_icetf", "pool->E")])]
for row, cells in GRID:
    for arm, lab in cells:
        p = Path(OUT_DIR) / f"ft4var_w9_wcle_{arm}_fp_best.json"
        if not p.exists():
            print(f"{row} {lab:9s} (missing)"); continue
        d = json.loads(p.read_text())
        runs = d["per_seed"]
        r = {v: np.mean([x[v]["h1"] for x in runs]) for v in VORD}
        print(f"{row} {lab:9s} ep{d.get('best_ep'):>4} "
              + " ".join(f"{v[:3]}:{r[v]:.3f}" for v in VORD)
              + f" m4:{np.mean(list(r.values())):.3f} "
              f"tag:{np.mean([x['noname']['tag'] for x in runs]):.3f}")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
